# Munti — ablation: no positional embeddings

Identical to the main run in every respect except `learned_pos: false`. Same
seed, same steps, same LR schedule — and, by mounting the main run's output,
the **same token stream and the same tokenizer**. Retokenizing would introduce
a second difference and weaken the claim that the loss gap comes from removing
positional information.

Attention is a weighted sum, so it's order-blind on its own. Positional
embeddings are the only thing telling the model where a token sits. Removing
them should leave a model that knows *which* words tend to co-occur but not in
what order.

In [ ]:
import os, shutil, sys, glob
from pathlib import Path

WORK = Path("/kaggle/working/munti-repo")
if not WORK.exists():
    # Prefer the dataset. Mounted kernel outputs each contain a full copy of the
    # repo too, so a bare "first pyproject.toml anywhere" grabs whichever stale
    # snapshot the glob happens to hit first.
    roots = [Path(p).parent for p in glob.glob("/kaggle/input/munti-source/**/pyproject.toml", recursive=True)]
    roots += [Path(p).parent for p in glob.glob("/kaggle/input/*/**/pyproject.toml", recursive=True)]
    assert roots, "repo not found in /kaggle/input"
    shutil.copytree(roots[0], WORK)
os.chdir(WORK); sys.path.insert(0, str(WORK))

# Reuse the main run's token stream. Deliberately NOT its out/ directory — that
# holds the baseline checkpoint, and copying it would make this resume the
# baseline instead of training a fresh no-positions model.
(WORK / "data").mkdir(exist_ok=True)
for name in ("train.bin", "val.bin", "tokenizer.json"):
    src = next((p for p in glob.glob(f"/kaggle/input/*/**/data/{name}", recursive=True)), None)
    assert src, f"{name} not found — attach the munti-train kernel output"
    shutil.copy(src, WORK / "data" / name)
assert not (WORK / "out-nopos" / "ckpt.pt").exists(), "stale ablation checkpoint"

import torch
cap = torch.cuda.get_device_capability()
print("gpu:", torch.cuda.get_device_name(0), f"sm_{cap[0]}{cap[1]}")
assert cap >= (7, 0), "needs sm_70+; push with --accelerator NvidiaTeslaT4"
print("reused token stream:", (WORK / "data" / "train.bin").stat().st_size // 2**20, "MB")

In [ ]:
from munti.train import train, plot_curve

train("configs/ablation-nopos.yaml", resume=False)
plot_curve("out-nopos/loss.csv")

In [ ]:
# Same prompts as the baseline, so the two can be read side by side.
import torch
from munti.model import Munti
from munti.sample import generate_text
from munti import tokenizer as tk

model = Munti.from_checkpoint(torch.load("out-nopos/ckpt.pt", map_location="cuda", weights_only=False), device="cuda")
tok = tk.load()
for prompt in [
    "Once upon a time, there was a little girl named Lily.",
    "Tom found a shiny red box under the tree. He",
    "The cat was hungry, so",
]:
    print("-" * 70)
    print(generate_text(model, tok, prompt, device="cuda", max_new_tokens=200, temperature=0.8, top_k=200))

In [ ]:
shutil.make_archive("/kaggle/working/munti-ablation", "zip", "out-nopos")
print(sorted(os.listdir("out-nopos")))